FRAMEWORK TO CONVERT A GENERAL CNN TO A NETWORK CONTAINING ONLY FULLY CONNECTED LAYERS

At the current state the original network has to have only layer which contain (InputLayer, FC, Conv2d, MaxPool2d, BatchNormalization, Flatten) layers. As example, it will used a network having a batch size of 32 and with the structure:
 INPUT (40, 40, 1) -> BATCH NORMALIZATION -> CONV2D -> MAXPOOL2D -> CONV2D -> MAXPOOL2D -> CONV2D -> CONV2D -> FC -> FC (classification with softmax activation)

The network above will be converted into an equal one with the following structure:

|   Original Layer  |   Output Shape |   FC Neurons  |   Method   |
-------------------------------------------------------------------
|   1. Input        |   (40,40,1)    |       -       |   Flatten  |
|   2. BN + Conv2D  |   (40,40,8)    |     12800     | Direct FC  |
|   3. MaxPool2D    |   (13,13,8)    |      6400     |  Distill   |
|   4. Conv2D       |   (13,13,16)   |      2704     | Direct FC  |
|   5. MaxPool2D    |   (6,6,16)     |      1352     |  Distill   |
|   6. Conv2D       |   (3,3,32)     |      288      | Direct FC  |
|   7. Conv2D       |   (2,2,64)     |      256      | Direct FC  |
|   8. Flatten      |   (256)        |      256      |  Flatten   |
|   9. FC           |   (128)        |      128      | Already FC |
|   10. FC (output) |   (94)         |      94       | Already FC |
-------------------------------------------------------------------

[1] Train, test and validation data split and uploading of the teacher model. Threshold parameter passed in create_dataset function is at most 1448 for the dataset provided and it indicates that for each class there will be 1448 samples, this is done to have a balanced dataset per class

In [1]:
import os
import gc
import numpy as np
import tensorflow as tf
from tensorflow.keras import Model, layers
from sklearn.model_selection import train_test_split

#1. Set random seed
seed=22
tf.random.set_seed(seed)
np.random.seed(seed)

def create_dataset(shape_input, file, values_name, labels_name, threshold):
    "The file format supported is a .npz"
    if len(shape_input)==3:
        height, width, channels=shape_input
    elif len(shape_input)==2:
        height, width=shape_input
        channels=1
    else:
        raise ValueError("Shape Input must be (height, width) or (height, width, channels)")
    
    # 2. Load dataset
    dataset = np.load(file)
    samples = dataset[values_name][:, :height, :width]
    classes = dataset[labels_name].astype(int)
    
    # 3. Filter / balance classes
    filtered_samples, filtered_classes = [], []
    unique_classes, counts = np.unique(classes, return_counts=True)
    if threshold != -1:
        keep_classes = unique_classes[counts >= threshold]
    else:
        keep_classes = unique_classes
    
    for cls in keep_classes:
        cls_indices = np.where(classes == cls)[0]
        if threshold != -1:
            selected_indices = np.random.choice(cls_indices, size=threshold, replace=False)
        else:
            selected_indices = cls_indices
        filtered_samples.append(samples[selected_indices])
        filtered_classes.append(classes[selected_indices])
    
    filtered_samples = np.concatenate(filtered_samples)
    filtered_classes = np.concatenate(filtered_classes)
    
    # 4. Normalize class indices
    unique_classes = np.unique(filtered_classes)
    class_mapping = {cls: idx for idx, cls in enumerate(unique_classes)}
    filtered_classes = np.array([class_mapping[cls] for cls in filtered_classes])
    
    # 5. Split into train/val/test
    X_train, X_temp, y_train, y_temp = train_test_split(
        filtered_samples, filtered_classes, test_size=0.3, random_state=seed)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=seed)
    
    # Expand dims if grayscale input
    if len(shape_input) == 2:
        X_train = np.expand_dims(X_train, axis=-1)
        X_val = np.expand_dims(X_val, axis=-1)
        X_test = np.expand_dims(X_test, axis=-1)
    
    print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
    print(f"Number of classes: {len(unique_classes)}")
    return X_train, y_train, X_val, y_val, X_test, y_test

# Splitting Dataset Generation
shape_input=(40,40,1)
X_train, y_train, X_val, y_val, X_test, y_test = create_dataset(
    shape_input, file="sheila-mfe-1sec.npz", 
    values_name="features", labels_name="labels", threshold=-1)

print("\n" + "="*50)
print("FIXING DATASET SHAPE FOR MODEL COMPATIBILITY")
print("="*50)

print("Before reshaping:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_test: {X_test.shape}")

# Add the channel dimension explicitly
X_train = np.expand_dims(X_train, axis=-1)
X_val = np.expand_dims(X_val, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

print("\nAfter reshaping:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_test: {X_test.shape}")

# 6. Load teacher model, that extracts only d-vector
teacher_dvec_model = tf.keras.models.load_model("kws_cnn_teacher.h5")
teacher_dvec_model.summary()

Train: (13004, 40, 40), Val: (2787, 40, 40), Test: (2787, 40, 40)
Number of classes: 2

FIXING DATASET SHAPE FOR MODEL COMPATIBILITY
Before reshaping:
  X_train: (13004, 40, 40)
  X_val: (2787, 40, 40)
  X_test: (2787, 40, 40)

After reshaping:
  X_train: (13004, 40, 40, 1)
  X_val: (2787, 40, 40, 1)
  X_test: (2787, 40, 40, 1)


2025-08-29 19:22:04.243664: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2025-08-29 19:22:04.243873: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2025-08-29 19:22:04.243889: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2025-08-29 19:22:04.244131: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-08-29 19:22:04.244149: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "teacher_kws_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 40, 40, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_58 (Conv2D)              │ (None, 40, 40, 32)     │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_99          │ (None, 40, 40, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_99 (ReLU)                 │ (None, 40, 40, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_57 (MaxPooling2D) │ (None, 20, 20, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_114 (Dropout)           │ (None, 20, 20, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_59 (Conv2D)              │ (None, 20, 20, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_100         │ (None, 20, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_100 (ReLU)                │ (None, 20, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_58 (MaxPooling2D) │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_115 (Dropout)           │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_60 (Conv2D)              │ (None, 10, 10, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_101         │ (None, 10, 10, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_101 (ReLU)                │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_59 (MaxPooling2D) │ (None, 5, 5, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_116 (Dropout)           │ (None, 5, 5, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_17 (Flatten)            │ (None, 3200)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_76 (Dense)                │ (None, 256)            │       819,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_102         │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_102 (ReLU)                │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_117 (Dropout)           │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_77 (Dense)                │ (None, 128)            │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_103         │ (None, 128)            │           51

 Total params: 955,428 (3.64 MB)

 Trainable params: 954,082 (3.64 MB)

 Non-trainable params: 1,344 (5.25 KB)

 Optimizer params: 2 (12.00 B)

[2] Libraries that may not be installed and that are required for this blocks to be run

In [18]:
pip install tqdm matplotlib


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


[3] Definition of a custom class called SoftMaxPooling2D that performs and approximation of pooling layers into a linear function

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Layer
import math

class SoftMaxPooling2D(Layer):
    """
    Custom Layer that approximates MaxPooling2D and dynamically adjusts pooling parameters.
    """
    def __init__(self, target_output_shape=None, pool_size=(2,2), alpha=100.0, **kwargs):
        super(SoftMaxPooling2D, self).__init__(**kwargs)
        self.target_output_shape = target_output_shape
        self.pool_size = pool_size
        self.alpha = alpha

    def call(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, h, w, c = input_shape[0], input_shape[1], input_shape[2], input_shape[3]

        if self.target_output_shape is not None:
            out_h, out_w = self.target_output_shape
        else:
            pool_h, pool_w = self.pool_size
            out_h = h // pool_h
            out_w = w // pool_w
            if out_h == 0 and h == 1:
                out_h = 1
            if out_w == 0 and w == 1:
                out_w = 1

        h_crop = out_h * (h // out_h if out_h > 0 else 1)
        w_crop = out_w * (w // out_w if out_w > 0 else 1)
        
        inputs_cropped = inputs[:, :h_crop, :w_crop, :]
        
        pool_h_calc = h_crop // out_h
        pool_w_calc = w_crop // out_w
        
        inputs_reshaped = tf.reshape(inputs_cropped, [batch_size, out_h, pool_h_calc, out_w, pool_w_calc, c])
        inputs_permuted = tf.transpose(inputs_reshaped, perm=[0, 1, 3, 5, 2, 4])
        inputs_windows = tf.reshape(inputs_permuted, [batch_size, out_h, out_w, c, pool_h_calc * pool_w_calc])
        
        scaled_inputs = inputs_windows * self.alpha
        max_vals = tf.reduce_max(scaled_inputs, axis=-1, keepdims=True)
        shifted_inputs = scaled_inputs - max_vals
        exp_shifted = tf.exp(shifted_inputs)
        sum_exp = tf.reduce_sum(exp_shifted, axis=-1, keepdims=True)
        log_sum_exp = max_vals + (1.0 / self.alpha) * tf.math.log(sum_exp)
        
        return tf.squeeze(log_sum_exp, axis=-1)

    def get_config(self):
        config = super(SoftMaxPooling2D, self).get_config()
        config.update({
            'target_output_shape': self.target_output_shape,
            'pool_size': self.pool_size,
            'alpha': self.alpha
        })
        return config

    def compute_output_shape(self, input_shape):
        batch_size, h, w, c = input_shape
        if self.target_output_shape is not None:
            return (batch_size, self.target_output_shape[0], self.target_output_shape[1], c)
        else:
            out_h = h // self.pool_size[0] if h is not None else None
            out_w = w // self.pool_size[1] if w is not None else None
            if out_h == 0 and h is not None and h == 1:
                out_h = 1
            if out_w == 0 and w is not None and w == 1:
                out_w = 1
            return (batch_size, out_h, out_w, c)

[4] Custom class SparseDense that performs the essential connections to mimic a convolution behavior without having all the inputs connected to the subsequent output in FullyConnected layers

In [3]:
import tensorflow as tf
from tensorflow.keras.layers import Layer
import numpy as np
from scipy import sparse

class SparseDense(Layer):
    def __init__(self, units, pattern=None, **kwargs):
        super(SparseDense, self).__init__(**kwargs)
        self.units=units
        self.pattern=pattern
        
    def build(self, input_shape):
        if self.pattern is not None:
            self.non_zero_weights=self.add_weight(
                name="non_zero_weights",
                shape=(len(self.pattern['values']),),
                initializer=tf.constant_initializer(self.pattern['values']),
                trainable=True
            )
        else:
            self.dense=tf.keras.layers.Dense(self.units)
        
        self.bias=self.add_weight(
            name='bias',
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
    
    def call(self, inputs):
        if self.pattern is not None:
            sparse_weights=tf.SparseTensor(
                indices=self.pattern['indices'],
                values=self.non_zero_weights,
                dense_shape=[int(inputs.shape[-1]), self.units]
            ) 
            output=tf.sparse.sparse_dense_matmul(inputs, sparse_weights)
            return output+self.bias
        else:
            return self.dense(inputs)+self.bias
        
    def get_config(self):
        config=super(SparseDense, self).get_config()
        config.update({
            'units': self.units,
            'pattern': self.pattern
        })
        return config
    
    @classmethod
    def from_config(cls, config):
        return cls(**config)

[5] Actual training from teacher model to intermediate neural network

In [ ]:

import math
import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Activation, Dense, Dropout, Flatten, BatchNormalization, Conv2D, MaxPooling2D, InputLayer, Reshape 
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import CosineSimilarity
import tensorflow.keras.backend as K

def cosine_similarity_loss(y_true, y_pred):
    "Cosine similarity loss for direction"
    y_true=K.l2_normalize(y_true, axis=-1)
    y_pred=K.l2_normalize(y_pred, axis=-1)
    return 1-K.sum(y_true*y_pred, axis=-1)

def magnitude_loss(y_true, y_pred):
    "MSE loss for magnitude"
    mag_true=tf.norm(y_true, axis=-1)+1e-8
    mag_pred=tf.norm(y_pred, axis=-1)+1e-8
    log_ratio=tf.math.log(mag_pred/mag_true)
    return tf.square(log_ratio)

def combined_representation_loss(y_true, y_pred, alpha=0.5):
    "Combined loss matching"
    cosine_loss=cosine_similarity_loss(y_true, y_pred)
    mag_loss=magnitude_loss(y_true, y_pred)
    return alpha*cosine_loss+(1-alpha)*mag_loss

def task_classification_loss(y_true, y_pred):
    "Categorical for task performance"
    y_pred=tf.clip_by_value(y_pred, 1e-7, 1.0-1e-7)
    return tf.keras.losses.categorical_crossentropy(y_true, y_pred)

class GlobalDistillationTrainer:
    def __init__(self, teacher_model, input_shape):
        "Class Initialization"
        print("="*80)
        print("INITIALIZING GLOBAL DISTILLATION")
        print("="*80)
        self.teacher_model=teacher_model
        self.input_shape=input_shape
        self.input_size=np.prod(input_shape)
        self.num_classes=-1
        self.teacher_layers={}
        self.student_model=None
        self.layer_mapping={}
        self.teacher_feature_extractors={}
        print(f"Teacher model provided: {teacher_model is not None}")
        print(f"Input shape: {self.input_shape}")
        print(f"Flattened input: {self.input_size}")
        print("\nStarting teacher model analysis")
        self.analyze_teacher_model()
        print("Initialization completed successfully!")
    
    def analyze_teacher_model(self):
        "Extract layer into from teacher"
        print("\n"+"="*80)
        print("ANALYZING TEACHER MODEL ARCHITECTURE")
        print("="*80)
        print("Teacher model summary:")
        try:
            self.teacher_model.summary()
            print("Model summary displayed successfully!")
        except Exception as e:
            print(f"Error displaying teacher model: {e}")
            
        print("\nVerification if there are only supported layers\n")
        print(f"Model contains {len(self.teacher_model.layers)} layers")
        
        teacher_layers=[]
        intermediate_outputs=[]

        for i, layer in enumerate(self.teacher_model.layers):
            layer_type=type(layer).__name__
            print(f" Layer {i:2d}: {layer_type:20s} - {layer.name}")
            if isinstance(layer, (Dense, Conv2D, MaxPooling2D, BatchNormalization)):
                output_shape=layer.output.shape
                output_size=np.prod(output_shape[1:]) if len(output_shape)>1 else 1
                layer_info={
                    'index': i,
                    'name': layer.name,
                    'type': layer_type,
                    'layer_obj': layer,
                    'output_shape': output_shape,
                    'output_size': output_size
                }
                if hasattr(layer, 'filters'):
                    layer_info['filters']=layer.filters
                if hasattr(layer, 'kernel_size'):
                    layer_info['kernel_size']=layer.kernel_size
                if hasattr(layer, 'strides'):
                    layer_info['strides']=layer.strides
                if hasattr(layer, 'padding'):
                    layer_info['padding']=layer.padding
                    
                teacher_layers.append(layer_info)
                intermediate_outputs.append(layer.output)
                
                self.teacher_feature_extractors[i]=Model(
                    inputs=self.teacher_model.input,
                    outputs=layer.output,
                    name=f"teacher_extractor_{i}"
                )
                
        self.teacher_layers=teacher_layers
        for layer in self.teacher_model.layers:
            layer.trainable=False
        
    def create_layer_mapping(self):
        "Create proper mapping between teacher and student layers"
        print("\n" + "-" * 80)
        print("CREATING LAYER MAPPING FOR DISTILLATION")
        print("-" * 80)
        
        self.layer_mapping = {}
        
        # Group teacher layers by type
        teacher_conv_layers = [l for l in self.teacher_layers if l['type'] == 'Conv2D']
        teacher_bn_layers = [l for l in self.teacher_layers if l['type'] == 'BatchNormalization']
        teacher_dense_layers = [l for l in self.teacher_layers if l['type'] == 'Dense']
        
        # Group student layers by type
        student_sparse_layers = [i for i, layer in enumerate(self.student_model.layers) 
                            if isinstance(layer, SparseDense)]
        student_bn_layers = [i for i, layer in enumerate(self.student_model.layers) 
                        if isinstance(layer, BatchNormalization)]
        student_dense_layers = [i for i, layer in enumerate(self.student_model.layers) 
                            if isinstance(layer, Dense) and 'softmax' not in layer.name]
        student_final_dense = [i for i, layer in enumerate(self.student_model.layers) 
                            if isinstance(layer, Dense) and 'softmax' in layer.name]
        
        print(f"Teacher Conv2D layers: {[l['index'] for l in teacher_conv_layers]}")
        print(f"Teacher BN layers: {[l['index'] for l in teacher_bn_layers]}")
        print(f"Teacher Dense layers: {[l['index'] for l in teacher_dense_layers]}")
        print(f"Student Sparse layers: {student_sparse_layers}")
        print(f"Student BN layers: {student_bn_layers}")
        print(f"Student Dense layers: {student_dense_layers}")
        print(f"Student Final Dense: {student_final_dense}")
        
        # Map Conv2D -> SparseDense
        for teacher_conv, student_sparse in zip(teacher_conv_layers, student_sparse_layers):
            self.layer_mapping[teacher_conv['index']] = student_sparse
            print(f"    Conv2D {teacher_conv['index']} -> SparseDense {student_sparse}")
        
        # Map BatchNormalization -> BatchNormalization
        for teacher_bn, student_bn in zip(teacher_bn_layers, student_bn_layers):
            self.layer_mapping[teacher_bn['index']] = student_bn
            print(f"    BN {teacher_bn['index']} -> BN {student_bn}")
        
        # Map Dense -> Dense (except final layer)
        for teacher_dense, student_dense in zip(teacher_dense_layers[:-1], student_dense_layers):
            self.layer_mapping[teacher_dense['index']] = student_dense
            print(f"    Dense {teacher_dense['index']} -> Dense {student_dense}")
        
        # Map final dense layer
        if teacher_dense_layers and student_final_dense:
            self.layer_mapping[teacher_dense_layers[-1]['index']] = student_final_dense[0]
            print(f"    Final Dense {teacher_dense_layers[-1]['index']} -> Final Dense {student_final_dense[0]}")
            
    def create_dense_student(self, dropout_rate=0.3):
        "Create dense neural network with number of layers equal to CNN"
        print("\n"+"-"*80)
        print("CREATING STUDENT DENSE NETWORK")
        print("-"*80)
        
        input_layer=Input(shape=(self.input_size,), name="input")
        x=input_layer
        self.student_intermediate_outputs={}
        for i, layer in enumerate(self.teacher_layers):
            output_size=layer['output_size']
            output_shape=layer['output_shape']
            match layer['type']:
                case 'InputLayer':
                    pass
                case 'Dense':
                    if i==len(self.teacher_layers)-1:
                        x=Dense(output_size, activation="softmax", name=f"dense_{i}_softmax")(x)
                        self.num_classes=output_size
                    else:
                        x=Dense(output_size, activation="relu", name=f"dense_{i}_relu")(x)
                        self.student_intermediate_outputs[layer['index']]=x
                        x=Dropout(dropout_rate, name=f"dropout_{i}")(x)
                case 'Conv2D':
                    #if layer['layer_obj'].strides==(1,1):
                    #if layer['strides']==(1,1):
                    if not layer['layer_obj'].use_bias:
                        prev_layer=self.teacher_layers[i-1]
                        prev_shape=prev_layer['output_shape']
                        if len(prev_shape)==2:
                            prev_shape=(40, 40, 1)
                        print(prev_shape)
                        pattern=self.conv2d_to_dense_weights(layer['layer_obj'], prev_shape)
                        x=SparseDense(units=output_size, pattern=pattern, name=f"sparse_{i}_conv")(x)
                        self.student_intermediate_outputs[layer['index']]=x
                        x=BatchNormalization(name=f"bn_{i}")(x)
                        x=Activation('relu', name=f"relu_{i}")(x)
                    else:
                        pattern = self.conv2d_to_dense_weights(layer['layer_obj'], current_shape)
                        x = SparseDense(units=output_size, pattern=pattern, name=f"sparse_{i}_conv")(x)
                        self.student_intermediate_outputs[layer['index']] = x
                case 'MaxPooling2D':
                    prev_layer=self.teacher_layers[i-1]
                    prev_shape=prev_layer['output_shape']
                    if len(prev_shape)>=4:
                        _, h_in, w_in, c_in=prev_shape
                        _, h_out, w_out, c_out=output_shape
                    else:
                        h_in, w_in, c_in=prev_shape
                        h_out, w_out, c_out=output_shape
                    
                    x=Reshape((h_in, w_in, c_in), name=f"reshape_to_pool_{i}")(x)
                    x=SoftMaxPooling2D(target_output_shape=(h_out, w_out), alpha=10.0, name=f"soft_max_pool_{i}")(x)
                    x=Flatten(name=f"flatten_{i}")(x)
                    #x=Dense(output_size, activation="relu", name=f"dense_{i}_pool")(x)
                    #x=Dropout(dropout_rate, name=f"dropout_{i}")(x)
                case 'BatchNormalization':
                    x= BatchNormalization(name=f"bn_{i}", trainable=False)(x)
                case 'Flatten' | 'Dropout':
                    continue
                case _:
                    print(f"Warning: Unsupported layer type {layer['type']}")
            current_shape = layer['output_shape'][1:]
            
        student_model=Model(
            inputs=input_layer,
            outputs=x,
            name="FullyConnectedModel"
        )
        
        sample_input = np.zeros((1, self.input_size))
        _=student_model(sample_input)
        self.student_model=student_model
        student_model.summary()
        self.create_layer_mapping()
        return self.student_model
    """
    def set_mathematical_weights(self, student_model):
        "Set weights from teacher CNN to student Dense"
        print("\n" + "=" * 80)
        print("SETTING WEIGHTS WITH MATH METHOD")
        print("=" * 80)

        for teacher_layer_info in self.teacher_layers:
            teacher_idx = teacher_layer_info['index']
            teacher_type = teacher_layer_info['type']
            teacher_layer = teacher_layer_info['layer_obj']
            
            # Only process layers that should be mapped and have weights
            if (teacher_idx in self.layer_mapping and 
                self.layer_mapping[teacher_idx] != -1 and
                teacher_type in ['Conv2D', 'Dense', 'BatchNormalization']):
                
                student_idx = self.layer_mapping[teacher_idx]
                student_layer = student_model.layers[student_idx]
                
                print(f"    Teacher Layer {teacher_idx} ({teacher_layer.name}) -> Student Layer {student_idx} ({student_layer.name})")
                
                try:
                    if teacher_type == 'Dense':
                        # Direct weight transfer for Dense layers
                        student_layer.set_weights(teacher_layer.get_weights())
                        print(f"    ✓ Set weights for {student_layer.name}")
                        
                    elif teacher_type == 'Conv2D':
                        # Convert Conv2D to equivalent dense weights
                        if isinstance(student_layer, SparseDense):
                            # Get input shape from previous layer
                            prev_layer_idx = student_idx - 1
                            if prev_layer_idx >= 0:
                                prev_output_shape = student_model.layers[prev_layer_idx].output_shape
                                if len(prev_output_shape) == 2:
                                    # Flattened input, need to reshape
                                    input_size = prev_output_shape[1]
                                    # Estimate original shape (simplified)
                                    sqrt_size = int(math.sqrt(input_size))
                                    input_shape = (sqrt_size, sqrt_size, 1)
                                else:
                                    input_shape = prev_output_shape[1:]
                                
                                weight_info = self.conv2d_to_dense_weights(teacher_layer, input_shape)
                                # Set sparse weights if your SparseDense layer supports it
                                if hasattr(student_layer, 'set_sparse_weights'):
                                    student_layer.set_sparse_weights(weight_info)
                                    print(f"    ✓ Set sparse weights for {student_layer.name}")
                        
                    elif teacher_type == 'BatchNormalization':
                        # Transfer BatchNorm parameters
                        student_layer.set_weights(teacher_layer.get_weights())
                        print(f"    ✓ Set BatchNorm weights for {student_layer.name}")
                        
                except Exception as e:
                    print(f"    ✗ Error setting weights for {teacher_layer.name}: {e}")
                    import traceback
                    traceback.print_exc()
    """  
    def conv2d_to_dense_weights(self, conv_layer, input_shape, next_layer=None):
        """Convert Conv2D layer weights to equivalent Dense layer weights (vectorized)."""
        print(f"\nConverting {conv_layer.name} to Dense layer")
        
        # Handle the no-bias case
        layer_weights = conv_layer.get_weights()
        
        if len(layer_weights) == 2:
            # Has both weights and biases
            weights, biases = layer_weights
            has_bias = True
        elif len(layer_weights) == 1:
            # Only has weights, no biases (use_bias=False)
            weights = layer_weights[0]
            biases = None
            has_bias = False
            print(f"  Layer {conv_layer.name} has no bias (use_bias=False)")
        else:
            raise ValueError(f"Unexpected number of weight arrays: {len(layer_weights)}")
        
        strides = conv_layer.strides
        kernel_h, kernel_w, in_ch, out_ch = weights.shape
        
        if len(input_shape) == 4:
            _, in_h, in_w, in_c = input_shape
        elif len(input_shape) == 3:
            in_h, in_w, in_c = input_shape
        else:
            in_h, in_w = input_shape
            in_c = 1
            
        if in_c != in_ch:
            raise ValueError(f"Input channels mismatch: expected {in_ch}, got {in_c}")
        
        # Calculate output dimensions and padding
        if conv_layer.padding.lower() == 'same':
            out_h = math.ceil(in_h / strides[0])
            out_w = math.ceil(in_w / strides[1])
            pad_y = (out_h * strides[0] - in_h + kernel_h - strides[0])
            pad_x = (out_w * strides[1] - in_w + kernel_w - strides[1])
            pad_top = pad_y // 2
            pad_bottom = pad_y - pad_top
            pad_left = pad_x // 2
            pad_right = pad_x - pad_left
        else:
            out_h = math.ceil((in_h - kernel_h) / strides[0]) + 1
            out_w = math.ceil((in_w - kernel_w) / strides[1]) + 1
            pad_top = pad_bottom = pad_left = pad_right = 0
        
        print(f"  Conv params: kernel={kernel_h}x{kernel_w}, in_ch={in_ch}, out_ch={out_ch}")
        print(f"  Input: {in_h}x{in_w}x{in_c}")
        print(f"  Output: {out_h}x{out_w}x{out_ch}")
        print(f"  Has bias: {has_bias}")
        
        input_size = in_h * in_w * in_c
        output_size = out_h * out_w * out_ch
        
        # Handle bias creation
        if has_bias:
            if biases.shape != (out_ch,):
                raise ValueError(f"Unexpected bias shape: {biases.shape}, expected ({out_ch},)")
            dense_biases = np.tile(biases, out_h * out_w)
            print(f"  Using Conv2D biases")
        else:
            # No bias from Conv2D - check if next layer is BatchNormalization
            if next_layer and hasattr(next_layer, 'name') and 'batch_norm' in next_layer.name.lower():
                print(f"  No Conv2D bias - will use BatchNorm parameters from {next_layer.name}")
                # Get BatchNorm parameters: beta (bias), gamma (scale), moving_mean, moving_variance
                bn_weights = next_layer.get_weights()
                if len(bn_weights) == 4:
                    gamma, beta, moving_mean, moving_variance = bn_weights
                    # For BatchNorm: output = gamma * (input - moving_mean) / sqrt(moving_variance + epsilon) + beta
                    # The effective bias becomes: beta - gamma * moving_mean / sqrt(moving_variance + epsilon)
                    epsilon = next_layer.epsilon if hasattr(next_layer, 'epsilon') else 1e-3
                    effective_bias = beta - gamma * moving_mean / np.sqrt(moving_variance + epsilon)
                    dense_biases = np.tile(effective_bias, out_h * out_w)
                    print(f"  Created effective bias from BatchNorm: shape {dense_biases.shape}")
                else:
                    print(f"  Warning: Unexpected BatchNorm weights count: {len(bn_weights)}")
                    dense_biases = np.zeros(output_size, dtype=np.float32)
            else:
                # No bias at all
                dense_biases = np.zeros(output_size, dtype=np.float32)
                print(f"  No bias - using zeros")
        
        # Create weight mapping
        out_y_idx, out_x_idx = np.meshgrid(np.arange(out_h), np.arange(out_w), indexing='ij')
        out_y_idx = out_y_idx.flatten()
        out_x_idx = out_x_idx.flatten()
        
        start_y = out_y_idx * strides[0] - pad_top
        start_x = out_x_idx * strides[1] - pad_left
        
        indices = []
        values = []
        
        for ky in range(kernel_h):
            for kx in range(kernel_w):
                in_y = start_y + ky
                in_x = start_x + kx
                
                valid_mask = (in_y >= 0) & (in_y < in_h) & (in_x >= 0) & (in_x < in_w)
                if not np.any(valid_mask):
                    continue
                    
                in_flat = (in_y[valid_mask] * in_w + in_x[valid_mask]) * in_c
                out_flat_base = np.where(valid_mask)[0] * out_ch
                
                for ic in range(in_ch):
                    for oc in range(out_ch):
                        weight_value = weights[ky, kx, ic, oc]
                        for i in range(len(in_flat)):
                            indices.append([in_flat[i] + ic, out_flat_base[i] + oc])
                            values.append(weight_value)
        
        print(f"  Created weights: {len(indices)} connections")
        print(f"  Created biases shape: {dense_biases.shape}")
        
        return {
            'indices': np.array(indices, dtype=np.int64),
            'values': np.array(values, dtype=np.float32),
            'dense_shape': [input_size, output_size],
            'biases': dense_biases,
            'has_conv_bias': has_bias
        }

    def global_distillation_loss(self, student_outputs, teacher_outputs, y_true, layer_weights=None, alpha=0.5, task_weight=1.0):
        "Combination of Cosine Similarity, MSE and Task Classification"
        total_loss=0.0
        if layer_weights is None:
            layer_weights={idx:1.0 for idx in self.student_intermediate_outputs.keys()}
        representation_loss=0.0
        for teacher_idx in self.student_intermediate_outputs.keys():
            if teacher_idx in teacher_outputs and teacher_idx in student_outputs:
                teacher_repr=tf.reshape(teacher_outputs[teacher_idx], [tf.shape(teacher_outputs[teacher_idx])[0], -1])
                student_repr=tf.reshape(student_outputs[teacher_idx], [tf.shape(student_outputs[teacher_idx])[0], -1])
                layer_loss=combined_representation_loss(
                    teacher_repr,
                    student_repr,
                    alpha
                )
                weighted_loss=layer_weights.get(teacher_idx, 1.0)*tf.reduce_mean(layer_loss)
                representation_loss+=weighted_loss
        task_loss=task_classification_loss(y_true, student_outputs['final'])
        total_loss=representation_loss+task_weight*tf.reduce_mean(task_loss)
        return total_loss, representation_loss, tf.reduce_mean(task_loss)
    
    def plot_training_curves(self, history):
        """Plot comprehensive training and validation metrics with clear organization"""
        import matplotlib.pyplot as plt
        
        # Create figure with 2x2 subplots for better organization
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
        
        # Plot 1: Total Loss (main overview)
        ax1.plot(history['loss'], label='Training Loss', linewidth=2, color='blue')
        ax1.plot(history['val_loss'], label='Validation Loss', linewidth=2, color='red')
        ax1.set_title('Total Loss', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Accuracy (main performance metric)
        ax2.plot(history['accuracy'], label='Training Accuracy', linewidth=2, color='green')
        ax2.plot(history['val_accuracy'], label='Validation Accuracy', linewidth=2, color='orange')
        ax2.set_title('Accuracy', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        ax2.set_ylim(0, 1)  # Accuracy between 0-1
        
        # Plot 3: Breakdown of Loss Components
        ax3.plot(history['repr_loss'], label='Representation Loss', linewidth=1.5, color='purple', linestyle='--')
        ax3.plot(history['val_repr_loss'], label='Val Representation', linewidth=1.5, color='purple')
        ax3.plot(history['task_loss'], label='Task Loss', linewidth=1.5, color='brown', linestyle='--')
        ax3.plot(history['val_task_loss'], label='Val Task Loss', linewidth=1.5, color='brown')
        ax3.set_title('Loss Components Breakdown', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('Loss')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Loss Ratio (to monitor overfitting)
        train_ratio = np.array(history['repr_loss']) / np.array(history['task_loss'])
        val_ratio = np.array(history['val_repr_loss']) / np.array(history['val_task_loss'])
        ax4.plot(train_ratio, label='Train: Repr/Task Ratio', linewidth=2, color='magenta')
        ax4.plot(val_ratio, label='Val: Repr/Task Ratio', linewidth=2, color='cyan')
        ax4.axhline(y=1.0, color='gray', linestyle=':', alpha=0.7, label='Equal Ratio')
        ax4.set_title('Representation vs Task Loss Ratio', fontsize=14, fontweight='bold')
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('Ratio')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Additional: Create a separate summary plot
        self.plot_summary_metrics(history)

    def plot_summary_metrics(self, history):
        """Create a summary plot focusing on key metrics"""
        import matplotlib.pyplot as plt
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Left: Performance Metrics
        ax1.plot(history['accuracy'], label='Train Accuracy', color='green', linewidth=2)
        ax1.plot(history['val_accuracy'], label='Val Accuracy', color='darkgreen', linewidth=2)
        ax1.set_title('Model Performance', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Accuracy')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        ax1.set_ylim(0, 1)
        
        # Right: Loss Evolution
        epochs = range(1, len(history['loss']) + 1)
        ax2.semilogy(epochs, history['loss'], label='Total Loss', color='blue', linewidth=2)
        ax2.semilogy(epochs, history['repr_loss'], label='Repr Loss', color='red', linewidth=2, linestyle='--')
        ax2.semilogy(epochs, history['task_loss'], label='Task Loss', color='orange', linewidth=2, linestyle='--')
        ax2.set_title('Loss Evolution (Log Scale)', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss (log scale)')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print final metrics
        print(f"\n{'='*50}")
        print("FINAL TRAINING RESULTS SUMMARY")
        print(f"{'='*50}")
        print(f"Final Training Accuracy: {history['accuracy'][-1]:.4f}")
        print(f"Final Validation Accuracy: {history['val_accuracy'][-1]:.4f}")
        print(f"Final Total Loss: {history['loss'][-1]:.4f}")
        print(f"Final Representation Loss: {history['repr_loss'][-1]:.4f}")
        print(f"Final Task Loss: {history['task_loss'][-1]:.4f}")
        print(f"Best Validation Accuracy: {max(history['val_accuracy']):.4f} (epoch {np.argmax(history['val_accuracy']) + 1})")
        
    def _is_training_compatible_layer(self, layer):
        layer_type=type(layer).__name__
        training_compatible=[
            'Dense', 'Conv2D', 'Conv1D', 'Conv3D', 'BatchNormalization', 
            'Dropout', 'SpatialDropout1D', 'SpatialDropout2D', 'SpatialDropout3D',
            'LayerNormalization', 'GroupNormalization'
        ]
        return layer_type in training_compatible
    
    def _create_student_feature_extractors(self):
        self.student_feature_extractors={}
        for teacher_idx in self.student_intermediate_outputs.keys():
            if teacher_idx in self.layer_mapping:
                student_layer_idx=self.layer_mapping[teacher_idx]
                if student_layer_idx < len(self.student_model.layers):
                    outputs=self.student_model.layers[student_layer_idx].output
                    self.student_feature_extractors[teacher_idx]=Model(
                        inputs=self.student_model.input,
                        outputs=outputs,
                        name=f"student_extractor_{teacher_idx}"
                    )
                        
    def train_global_distillation(self, X_train, y_train, X_val, y_val, epochs=200, batch_size=32, learning_rate=1e-4, alpha=0.7, task_weight=1.0, layer_weights=None):
        "Global training with all layers unfrozen and multi-loss"
        print("\n"+"="*80)
        print("STARTING GLOBAL DISTILLATION TRAINING")
        print("="*80)
        self.create_dense_student()
        #self.set_mathematical_weights(self.student_model)
        self._create_student_feature_extractors()
        y_train_one=tf.keras.utils.to_categorical(y_train, num_classes=self.num_classes)
        y_val_one=tf.keras.utils.to_categorical(y_val, num_classes=self.num_classes)
        X_train_teacher=X_train
        X_val_teacher=X_val
        if len(X_train.shape)==3:
            X_train_teacher=np.expand_dims(X_train, -1)
            X_val_teacher=np.expand_dims(X_val, -1)
            
        X_train_student=X_train_teacher.reshape(X_train_teacher.shape[0], -1)
        X_val_student=X_val_teacher.reshape(X_val_teacher.shape[0], -1)
        optimizer=Adam(learning_rate=learning_rate)
        train_loss_metric=tf.keras.metrics.Mean(name='train_loss')
        val_loss_metric=tf.keras.metrics.Mean(name='val_loss')
        train_repr_loss_metric=tf.keras.metrics.Mean(name='train_repr_loss')
        val_repr_loss_metric=tf.keras.metrics.Mean(name='val_repr_loss')
        train_task_loss_metric=tf.keras.metrics.Mean(name='train_task_loss')
        val_task_loss_metric=tf.keras.metrics.Mean(name='val_task_loss')
        train_acc_metric=tf.keras.metrics.CategoricalAccuracy(name='train_accuracy')
        val_acc_metric=tf.keras.metrics.CategoricalAccuracy(name='val_accuracy')
        """
        @tf.function 
        def train_step(x_teacher, x_student, y_true):
            with tf.GradientTape() as tape:
                teacher_outputs={}
                for teacher_idx, extractor in self.teacher_feature_extractors.items():
                    teacher_outputs[teacher_idx]=extractor(x_teacher, training=False)
                student_predictions=self.student_model(x_student, training=True)
                student_outputs={}
                x=x_student
                
                for i, layer in enumerate(self.student_model.layers):
                    layer_type=type(layer).__name__ 
                    if layer_type in ['Dense', 'Conv2D']:
                        x=layer(x, training=True)
                    else:
                        x=layer(x)
                    for teacher_idx, student_layer_idx in self.layer_mapping.items():
                        if student_layer_idx==i and teacher_idx in self.student_intermediate_outputs:
                            student_outputs[teacher_idx]=x 
                student_outputs['final']=student_predictions
                total_loss, repr_loss, task_loss=self.global_distillation_loss(
                    student_outputs, teacher_outputs, y_true, layer_weights, alpha, task_weight
                )
            gradients=tape.gradient(total_loss, self.student_model.trainable_variables)
            gradients, _=tf.clip_by_global_norm(gradients, 5.0)
            optimizer.apply_gradients(zip(gradients, self.student_model.trainable_variables))
            train_loss_metric.update_state(total_loss)
            train_repr_loss_metric.update_state(repr_loss)
            train_task_loss_metric.update_state(task_loss)
            train_acc_metric.update_state(y_true, tf.nn.softmax(student_outputs['final']))
            return total_loss
        """
        @tf.function 
        def train_step(x_teacher, x_student, y_true):
            with tf.GradientTape() as tape:
                teacher_outputs = {}
                for teacher_idx, extractor in self.teacher_feature_extractors.items():
                    teacher_outputs[teacher_idx] = extractor(x_teacher, training=False)
                
                student_predictions = self.student_model(x_student, training=True)
                student_outputs = {}
                x = x_student
                for i, layer in enumerate(self.student_model.layers):
                    try:
                        x = layer(x, training=True)
                    except (TypeError, ValueError) as e:
                        try:
                            x = layer(x)
                        except Exception as e2:
                            continue
                    
                    # Store intermediate outputs for mapped layers
                    for teacher_idx, student_layer_idx in self.layer_mapping.items():
                        if student_layer_idx == i and teacher_idx in self.student_intermediate_outputs:
                            student_outputs[teacher_idx] = x
                
                student_outputs['final'] = student_predictions
                total_loss, repr_loss, task_loss = self.global_distillation_loss(
                    student_outputs, teacher_outputs, y_true, layer_weights, alpha, task_weight
                )
            
            gradients = tape.gradient(total_loss, self.student_model.trainable_variables)
            gradients=[tf.clip_by_value(g, -1.0, 1.0) for g in gradients]
            optimizer.apply_gradients(zip(gradients, self.student_model.trainable_variables))
            
            train_loss_metric.update_state(total_loss)
            train_repr_loss_metric.update_state(repr_loss)
            train_task_loss_metric.update_state(task_loss)
            train_acc_metric.update_state(y_true, tf.nn.softmax(student_outputs['final']))
            return total_loss
        """
        @tf.function 
        def val_step(x_teacher, x_student, y_true):
            teacher_outputs={}
            for teacher_idx, extractor in self.teacher_feature_extractors.items():
                teacher_outputs[teacher_idx]=extractor(x_teacher, training=False)
            
            student_outputs={}

            if not hasattr(self, '_student_feature_extractors'):
                self._student_feature_extractors = {}
                for teacher_idx in self.student_intermediate_outputs.keys():
                    if teacher_idx in self.layer_mapping.values():
                        self._student_feature_extractors[teacher_idx] = Model(
                            inputs=self.student_model.input,
                            outputs=self.student_model.layers[teacher_idx].output
                        )
            student_predictions=self.student_model(x_student, training=True)
            for teacher_idx, extractor in self._student_feature_extractor.items():
                student_outputs[teacher_idx]=extractor(x_student, training=True)
            student_outputs['final']=student_predictions
            
            total_loss, repr_loss, task_loss=self.global_distillation_loss(
                student_outputs, teacher_outputs, y_true, layer_weights, alpha, task_weight
            )
            val_loss_metric.update_state(total_loss)
            val_repr_loss_metric.update_state(repr_loss)
            val_task_loss_metric.update_state(task_loss)
            val_acc_metric.update_state(y_true, tf.nn.softmax(student_outputs['final']))
            return total_loss
        """ 
        @tf.function 
        def val_step(x_teacher, x_student, y_true):
            teacher_outputs = {}
            for teacher_idx, extractor in self.teacher_feature_extractors.items():
                teacher_outputs[teacher_idx] = extractor(x_teacher, training=False)
            
            student_outputs = {}
            x = x_student
            
            # Process through all layers (no training for validation)
            for i, layer in enumerate(self.student_model.layers):
                try:
                    x = layer(x, training=True)
                except (TypeError, ValueError) as e:
                    try:
                        x = layer(x)
                    except Exception as e2:
                        continue
                
                # Store intermediate outputs for mapped layers
                for teacher_idx, student_layer_idx in self.layer_mapping.items():
                    if student_layer_idx == i and teacher_idx in self.student_intermediate_outputs:
                        student_outputs[teacher_idx] = x
            
            student_outputs['final'] = x
            
            total_loss, repr_loss, task_loss = self.global_distillation_loss(
                student_outputs, teacher_outputs, y_true, layer_weights, alpha, task_weight
            )
            
            val_loss_metric.update_state(total_loss)
            val_repr_loss_metric.update_state(repr_loss)
            val_task_loss_metric.update_state(task_loss)
            val_acc_metric.update_state(y_true, tf.nn.softmax(student_outputs['final']))
            return total_loss
        
        print(f"Training for {epochs} epochs with batch size {batch_size}")
        print(f"Loss weights - Direction: {alpha}, Magnitude: {1-alpha}, Task: {task_weight}")
        history={
            'loss': [], 'val_loss': [],
            'repr_loss': [], 'val_repr_loss': [],
            'task_loss': [], 'val_task_loss': [],
            'accuracy': [], 'val_accuracy': []
        }
        best_val_loss=float("inf")
        patience_counter=0
        reduce_lr_counter=0
        patience=20
        reduce_lr_patience=10
        factor=0.5
        min_lr=1e-6
        best_weights=None
        
        for epoch in range(epochs):
            train_loss_metric.reset_state()
            val_loss_metric.reset_state()
            train_repr_loss_metric.reset_state()
            val_repr_loss_metric.reset_state()
            train_task_loss_metric.reset_state()
            val_task_loss_metric.reset_state()
            train_acc_metric.reset_state()
            val_acc_metric.reset_state()
            
            for i in range(0, len(X_train_teacher), batch_size):
                batch_x_teacher=X_train_teacher[i:i+batch_size]
                batch_x_student=X_train_student[i:i+batch_size]
                batch_y=y_train_one[i:i+batch_size]
                train_step(batch_x_teacher, batch_x_student, batch_y)
                
            for i in range(0, len(X_val_teacher), batch_size):
                batch_x_teacher=X_val_teacher[i:i+batch_size]
                batch_x_student=X_val_student[i:i+batch_size]
                batch_y=y_val_one[i:i+batch_size]
                val_step(batch_x_teacher, batch_x_student, batch_y)
            
            current_val_loss=float(val_loss_metric.result())
            if current_val_loss<best_val_loss:
                best_val_loss=current_val_loss
                patience_counter=0
                reduce_lr_counter=0
                best_weights=self.student_model.get_weights()
                print(" New best model found and saved")
            else:
                patience_counter+=1
                reduce_lr_counter+=1
                if reduce_lr_counter>=reduce_lr_patience:
                    old_lr=float(tf.keras.backend.get_value(optimizer.lr))
                    new_lr=max(old_lr*factor, min_lr)
                    tf.keras.backend.set_value(optimizer.lr, new_lr)
                    print(f"    LR reduced from {old_lr:.2e} to {new_lr:.2e}")
                    reduce_lr_counter=0
                if patience_counter>=patience:
                    print("Stopping triggered")
                    break
                 
            #if epoch%10==0 or epoch==epochs-1:
            print(f"Epoch {epoch+1}/{epochs}")
            print(f"    Loss: {train_loss_metric.result():.4f} | Val Loss: {val_loss_metric.result():.4f}")
            print(f"    Repr: {train_repr_loss_metric.result():.4f} | Val Repr: {val_repr_loss_metric.result():.4f}")
            print(f"    Task: {train_task_loss_metric.result():.4f} | Val Task: {val_task_loss_metric.result():.4f}")
            print(f"    Acc: {train_acc_metric.result():.4f} | Val Acc: {val_acc_metric.result():.4f}")
        
            history['loss'].append(float(train_loss_metric.result()))
            history['val_loss'].append(float(val_loss_metric.result()))
            history['repr_loss'].append(float(train_repr_loss_metric.result()))
            history['val_repr_loss'].append(float(val_repr_loss_metric.result()))
            history['task_loss'].append(float(train_task_loss_metric.result()))
            history['val_task_loss'].append(float(val_task_loss_metric.result()))
            history['accuracy'].append(float(train_acc_metric.result()))
            history['val_accuracy'].append(float(val_acc_metric.result()))
        
        print("\nGlobal distillation training completed")
        if best_weights is not None:
            self.student_model.set_weights(best_weights)
            print("Restored best model weights")
            
        self.plot_training_curves(history)
        self.student_model.save("student_model.h5")
        return self.student_model, history 

trained_model=GlobalDistillationTrainer(teacher_dvec_model, input_shape=(40,40,1))
student_model, history=trained_model.train_global_distillation(
    X_train, y_train, X_val, y_val, epochs=50, alpha=0.3, task_weight=2.0
)

INITIALIZING GLOBAL DISTILLATION
Teacher model provided: True
Input shape: (40, 40, 1)
Flattened input: 1600

Starting teacher model analysis

ANALYZING TEACHER MODEL ARCHITECTURE
Teacher model summary:


Model: "teacher_kws_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 40, 40, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_58 (Conv2D)              │ (None, 40, 40, 32)     │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_99          │ (None, 40, 40, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_99 (ReLU)                 │ (None, 40, 40, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_57 (MaxPooling2D) │ (None, 20, 20, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_114 (Dropout)           │ (None, 20, 20, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_59 (Conv2D)              │ (None, 20, 20, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_100         │ (None, 20, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_100 (ReLU)                │ (None, 20, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_58 (MaxPooling2D) │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_115 (Dropout)           │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_60 (Conv2D)              │ (None, 10, 10, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_101         │ (None, 10, 10, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_101 (ReLU)                │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_59 (MaxPooling2D) │ (None, 5, 5, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_116 (Dropout)           │ (None, 5, 5, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_17 (Flatten)            │ (None, 3200)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_76 (Dense)                │ (None, 256)            │       819,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_102         │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_102 (ReLU)                │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_117 (Dropout)           │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_77 (Dense)                │ (None, 128)            │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_103         │ (None, 128)            │           51

 Total params: 955,428 (3.64 MB)

 Trainable params: 954,082 (3.64 MB)

...

 Total params: 15,426,498 (58.85 MB)

 Trainable params: 14,887,106 (56.79 MB)

 Non-trainable params: 539,392 (2.06 MB)


--------------------------------------------------------------------------------
CREATING LAYER MAPPING FOR DISTILLATION
--------------------------------------------------------------------------------
Teacher Conv2D layers: [1, 6, 11]
Teacher BN layers: [2, 7, 12, 18, 22, 26]
Teacher Dense layers: [17, 21, 25, 29]
Student Sparse layers: [1, 8, 15]
Student BN layers: [2, 4, 9, 11, 16, 18, 24, 27, 30]
Student Dense layers: [22, 25, 28]
Student Final Dense: [31]
    Conv2D 1 -> SparseDense 1
    Conv2D 6 -> SparseDense 8
    Conv2D 11 -> SparseDense 15
    BN 2 -> BN 2
    BN 7 -> BN 4
    BN 12 -> BN 9
    BN 18 -> BN 11
    BN 22 -> BN 16
    BN 26 -> BN 18
    Dense 17 -> Dense 22
    Dense 21 -> Dense 25
    Dense 25 -> Dense 28
    Final Dense 29 -> Final Dense 31
Training for 50 epochs with batch size 32
Loss weights - Direction: 0.3, Magnitude: 0.7, Task: 2.0


2025-08-29 19:23:10.426004: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
